In [ ]:
import os, math, time, argparse, itertools, platform, random
from typing import Optional
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset
from transformers import (
    AutoTokenizer, BertForMaskedLM,
    DataCollatorForLanguageModeling, Trainer, TrainingArguments, set_seed
)

In [ ]:
def set_all_seeds(seed: int):
    set_seed(seed)
    random.seed(seed)
    import numpy as np
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def human(num):
    for unit in ["","K","M","B"]:
        if abs(num) < 1000:
            return f"{num:.1f}{unit}"
        num /= 1000
    return f"{num:.1f}T"

def print_gpu_mem(prefix=""):
    if not torch.cuda.is_available():
        return
    try:
        alloc = torch.cuda.max_memory_allocated() / 1e9
        reserv = torch.cuda.max_memory_reserved() / 1e9
        print(f"{prefix}Peak GPU memory -> allocated: {alloc:.2f} GB | reserved: {reserv:.2f} GB")
    except Exception:
        pass

# Data pipeline: WT103 subset
def build_wt103_subset(tokenizer, target_tokens=5_000_000, max_len=512, seed=1337):
    raw = load_dataset("wikitext", "wikitext-103-raw-v1")
    tok = tokenizer
    tok.model_max_length = 10_000_000

    taken_idx, seen = [], 0
    for i, line in enumerate(raw["train"]["text"]):
        if not line or not line.strip():
            continue
        seen += len(tok.tokenize(line))
        taken_idx.append(i)
        if seen >= target_tokens:
            break

    data = {
        "train": raw["train"].select(taken_idx),
        "validation": raw["validation"],
        "test": raw["test"],
    }

    def tok_fn(batch):
        return tok(
            batch["text"],
            add_special_tokens=False,
            truncation=False,
            padding=False,
            return_attention_mask=False,
            return_token_type_ids=False,
        )

    tokenized = {k: v.map(tok_fn, batched=True, remove_columns=["text"]) for k, v in data.items()}

    def group_and_add_specials(examples):
        all_ids = list(itertools.chain.from_iterable(examples["input_ids"]))
        eff = max_len - tok.num_special_tokens_to_add(pair=False)
        total = (len(all_ids) // eff) * eff
        if total == 0:
            return {"input_ids": [], "attention_mask": [], "token_type_ids": []}
        chunks = [all_ids[i:i+eff] for i in range(0, total, eff)]
        input_ids = [tok.build_inputs_with_special_tokens(c) for c in chunks]
        attention_mask = [[1]*len(x) for x in input_ids]
        token_type_ids = [[0]*len(x) for x in input_ids]
        return {"input_ids": input_ids, "attention_mask": attention_mask, "token_type_ids": token_type_ids}

    for split in ["train", "validation", "test"]:
        tokenized[split] = tokenized[split].map(
            group_and_add_specials,
            batched=True,
            remove_columns=tokenized[split].column_names,
        )

    tokenized["train"] = tokenized["train"].shuffle(seed=seed)
    return tokenized

# Local Wwindow sparse attention (chunked)
def _build_local_indices(L: int, w: int, device, global_stride: int, include_cls: bool):
    W = 2*w + 1
    base = torch.arange(L, device=device)[:, None]
    offsets = torch.arange(-w, w+1, device=device)[None, :]
    idx = (base + offsets).clamp(0, L-1)
    if include_cls:
        cls = torch.zeros(L, 1, dtype=idx.dtype, device=device)
        idx = torch.cat([cls, idx], dim=1)
    if global_stride and global_stride > 0:
        anchors = torch.arange(0, L, global_stride, device=device)[None, :].expand(L, -1)
        idx = torch.cat([idx, anchors], dim=1)
    return idx

class LocalWindowSelfAttention(nn.Module):
    """
    Drop-in for BertSelfAttention with *chunked* local-window attention.
    Avoids materializing (B,H,L,W,Hd) by processing sequence in chunks of size C.
    Supports global anchors to recover long-range context.
    """
    def __init__(self, config, window_size: int = 32, global_stride: int = 64,
                 include_cls: bool = True, chunk_size: int = 64):
        super().__init__()
        if window_size <= 0: raise ValueError("window_size must be > 0")
        if chunk_size <= 0: raise ValueError("chunk_size must be > 0")
        self.window_size  = int(window_size)
        self.global_stride = int(global_stride)
        self.include_cls  = bool(include_cls)
        self.chunk_size   = int(chunk_size)

        self.num_attention_heads = config.num_attention_heads
        self.hidden_size = config.hidden_size
        self.attention_head_size = self.hidden_size // self.num_attention_heads
        self.all_head_size = self.num_attention_heads * self.attention_head_size

        self.query = nn.Linear(self.hidden_size, self.all_head_size)
        self.key   = nn.Linear(self.hidden_size, self.all_head_size)
        self.value = nn.Linear(self.hidden_size, self.all_head_size)

        self.dropout = nn.Dropout(config.attention_probs_dropout_prob)

        self._cached_idx = None
        self._cached_key = None

    def _shape_to_bhlh(self, x, B, L):
        H, Hd = self.num_attention_heads, self.attention_head_size
        return x.view(B, L, H, Hd).permute(0, 2, 1, 3).contiguous()

    def _ensure_indices(self, L, device):
        key = (L, self.window_size, device, self.global_stride, self.include_cls)
        if self._cached_key != key:
            self._cached_idx = _build_local_indices(
                L, self.window_size, device, self.global_stride, self.include_cls
            )
            self._cached_key = key
        return self._cached_idx  #

    def forward(
        self,
        hidden_states: torch.Tensor,
        attention_mask: Optional[torch.Tensor] = None,
        head_mask=None,
        encoder_hidden_states=None,
        encoder_attention_mask=None,
        past_key_value=None,
        output_attentions: bool=False,
        **kwargs
    ):
        del head_mask, encoder_hidden_states, encoder_attention_mask, past_key_value, kwargs
        B, L, D = hidden_states.shape
        H, Hd = self.num_attention_heads, self.attention_head_size
        idx = self._ensure_indices(L, hidden_states.device)
        Wg = idx.size(1)


        q_lin = self.query(hidden_states)
        k_lin = self.key(hidden_states)
        v_lin = self.value(hidden_states)

        q = self._shape_to_bhlh(q_lin, B, L)
        k = self._shape_to_bhlh(k_lin, B, L)
        v = self._shape_to_bhlh(v_lin, B, L)

        if attention_mask is not None:
            pad_row = attention_mask.squeeze(1).squeeze(1)


        context = hidden_states.new_zeros(B, H, L, Hd)
        inv_sqrt = 1.0 / math.sqrt(Hd)
        C = max(1, self.chunk_size)

        # chunk along sequence to cap memory
        for s in range(0, L, C):
            e = min(s + C, L)
            q_s = q[:, :, s:e, :]
            idx_s = idx[s:e, :]

            expand_idx = idx_s[None, None, :, :, None].expand(B, H, e - s, Wg, Hd)
            k_win = torch.take_along_dim(k.unsqueeze(3), expand_idx, dim=2)
            v_win = torch.take_along_dim(v.unsqueeze(3), expand_idx, dim=2)

            # scores: einsum over Hd -> (B,H,C,Wg)
            scores = torch.einsum("bhlc,bhlwc->bhlw", q_s, k_win) * inv_sqrt

            if attention_mask is not None:
                pad_win = torch.take_along_dim(pad_row[:, None, :], idx_s[None, :, :], dim=-1)

            # softmax in window
            probs = F.softmax(scores, dim=-1)
            probs = self.dropout(probs)

            # context chunk
            ctx_s = torch.einsum("bhlw,bhlwc->bhlc", probs, v_win)
            context[:, :, s:e, :] = ctx_s

        # (B,L,D)
        context = context.permute(0, 2, 1, 3).contiguous().view(B, L, H*Hd)

        if output_attentions:
            return (context, None)
        return (context,)

def swap_in_local_attention(model: BertForMaskedLM, window_size: int = 32,
                            global_stride: int = 64, include_cls: bool = True,
                            chunk_size: int = 64):
    for layer in model.bert.encoder.layer:
        old = layer.attention.self
        new = LocalWindowSelfAttention(
            model.config,
            window_size=window_size,
            global_stride=global_stride,
            include_cls=include_cls,
            chunk_size=chunk_size,
        )
        # copy Q/K/V weights
        new.query.weight.data.copy_(old.query.weight.data)
        new.query.bias.data.copy_(old.query.bias.data)
        new.key.weight.data.copy_(old.key.weight.data)
        new.key.bias.data.copy_(old.key.bias.data)
        new.value.weight.data.copy_(old.value.weight.data)
        new.value.bias.data.copy_(old.value.bias.data)
        layer.attention.self = new
    return model

# main
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", type=str, default="bert-base-uncased")
    ap.add_argument("--max_len", type=int, default=512)
    ap.add_argument("--tokens", type=int, default=5_000_000)
    ap.add_argument("--steps", type=int, default=20_000)
    ap.add_argument("--batch_size", type=int, default=16)
    ap.add_argument("--seed", type=int, default=1337)

    ap.add_argument("--sparse", action="store_true", help="use local sparse attention")
    ap.add_argument("--window", type=int, default=32, help="local window size (±w -> 2w+1)")
    ap.add_argument("--global_stride", type=int, default=64, help="add global anchors every N positions (0=off)")
    ap.add_argument("--no_cls_global", action="store_true", help="do not add CLS (pos 0) as global key")
    ap.add_argument("--chunk", type=int, default=64, help="chunk size along L inside attention")

    ap.add_argument("--optimizer", type=str, default="adamw_torch",
                    choices=["adamw_torch", "adamw_bnb_8bit"],
                    help="8-bit reduces optimizer state memory (bitsandbytes required)")
    ap.add_argument("--try_8bit", action="store_true", help="attempt bitsandbytes 8-bit; fallback if unavailable")
    ap.add_argument("--grad_ckpt", action="store_true", help="enable gradient checkpointing")
    ap.add_argument("--compile", action="store_true", help="torch.compile the model if available")

    ap.add_argument("--out", type=str, default="./results/run")
    args, unknown = ap.parse_known_args()

    set_all_seeds(args.seed)
    print(f"PyTorch {torch.__version__} | CUDA {torch.cuda.is_available()} | Model {args.model}")
    print(f"Config: max_len={args.max_len} tokens≈{human(args.tokens)} steps={args.steps} "
          f"bs={args.batch_size} sparse={args.sparse} window={args.window} "
          f"gstride={args.global_stride} chunk={args.chunk} opt={args.optimizer} grad_ckpt={args.grad_ckpt}")

    tok = AutoTokenizer.from_pretrained(args.model)
    dset = build_wt103_subset(tok, target_tokens=args.tokens, max_len=args.max_len, seed=args.seed)
    print("Chunks → train:", len(dset["train"]), "val:", len(dset["validation"]))

    collator = DataCollatorForLanguageModeling(tokenizer=tok, mlm_probability=0.15)
    model = BertForMaskedLM.from_pretrained(args.model)
    if args.sparse:
        model = swap_in_local_attention(
            model,
            window_size=args.window,
            global_stride=args.global_stride,
            include_cls=not args.no_cls_global,
            chunk_size=args.chunk,
        )

    # CUDA
    torch.backends.cuda.matmul.allow_tf32 = True
    if hasattr(torch, "set_float32_matmul_precision"):
        torch.set_float32_matmul_precision("high")
    if torch.cuda.is_available():
        try: torch.cuda.reset_peak_memory_stats()
        except Exception: pass

    # gradient checkpointing
    if args.grad_ckpt:
        model.gradient_checkpointing_enable()

    if args.compile and hasattr(torch, "compile"):
        try:
            model = torch.compile(model, mode="max-autotune")
            print("torch.compile enabled.")
        except Exception as e:
            print(f"torch.compile not enabled: {e}")

    pin = torch.cuda.is_available()
    workers = 0 if platform.system().lower().startswith("win") else 2

    bf16_ok = torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8

    chosen_optim = args.optimizer
    if args.try_8bit or args.optimizer == "adamw_bnb_8bit":
        try:
            import bitsandbytes as bnb  # noqa: F401
            chosen_optim = "adamw_bnb_8bit"
        except Exception:
            chosen_optim = "adamw_torch"
            print("bitsandbytes not available; falling back to adamw_torch.")

    targs = TrainingArguments(
        output_dir=args.out,
        per_device_train_batch_size=args.batch_size,
        per_device_eval_batch_size=max(1, args.batch_size),
        gradient_accumulation_steps=1,
        max_steps=args.steps,
        learning_rate=5e-5,
        weight_decay=0.01,
        warmup_ratio=0.06,
        lr_scheduler_type="linear",

        eval_strategy="steps",
        eval_steps=1000,
        logging_strategy="steps",
        logging_steps=100,
        save_strategy="steps",
        save_steps=5000,
        save_total_limit=2,

        load_best_model_at_end=False,
        fp16=torch.cuda.is_available() and not bf16_ok,
        bf16=bf16_ok,
        report_to=[],
        seed=args.seed,
        remove_unused_columns=False,

        dataloader_num_workers=workers,
        dataloader_pin_memory=pin,
        group_by_length=True,

        include_tokens_per_second=True,
        include_num_input_tokens_seen=True,

        gradient_checkpointing=args.grad_ckpt,
        optim=chosen_optim,
        eval_accumulation_steps=2,
    )

    trainer = Trainer(
        model=model,
        args=targs,
        data_collator=collator,
        train_dataset=dset["train"],
        eval_dataset=dset["validation"],
    )

    start = time.time()
    trainer.train()
    wall = time.time() - start

    eval_res = trainer.evaluate()
    loss = float(eval_res["eval_loss"])
    ppl  = math.exp(loss) if loss < 20 else float("inf")

    seen = getattr(trainer.state, "num_input_tokens_seen", 0) or (args.steps * args.batch_size * args.max_len)
    toks_per_sec = seen / max(wall, 1e-6)

    print(f"Eval loss: {loss:.4f} | Perplexity: {ppl:.2f}")
    print(f"Time: {wall/60:.1f} min | Tokens/sec: {toks_per_sec:,.0f}")
    print_gpu_mem(prefix="Variant_1 ")
    if torch.cuda.is_available():
        try:
            print(f"Device memory total: {torch.cuda.get_device_properties(0).total_memory/1e9:.2f} GB")
        except Exception:
            pass

if __name__ == "__main__":
    os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
    os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
    main()


PyTorch 2.8.0+cu126 | CUDA True | Model bert-base-uncased
Config: max_len=512 tokens≈5.0M steps=20000 bs=16 sparse=False window=32 gstride=64 chunk=64 opt=adamw_torch grad_ckpt=False


Map:   0%|          | 0/52091 [00:00<?, ? examples/s]

Map:   0%|          | 0/3760 [00:00<?, ? examples/s]

Map:   0%|          | 0/4358 [00:00<?, ? examples/s]

Map:   0%|          | 0/52091 [00:00<?, ? examples/s]

...

Map:   0%|          | 0/4358 [00:00<?, ? examples/s]

Chunks → train: 9779 val: 466


Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Step,Training Loss,Validation Loss,Input Tokens Seen
1000,1.750800,1.596955,8185344
2000,1.661200,1.586738,16364032
3000,1.618800,1.563054,24549376
4000,1.564400,1.564965,32728064
5000,1.518400,1.525868,40906752
6000,1.489100,1.543238,49092096
7000,1.463600,1.509333,57270784
8000,1.433500,1.517571,65449472
9000,1.412300,1.550657,73634816
10000,1.382800,1.520717,81813504


Eval loss: 1.5228 | Perplexity: 4.59
Wall: 25.5 min | ~Tokens/sec: 106,839
[TRAIN] Peak GPU memory -> allocated: 7.88 GB | reserved: 9.08 GB
Device memory total: 42.47 GB
